Importando bibliotecas e Carregando bases

In [1]:
# Pandas será utilizado para o tratamento das bases

import pandas as pd
import numpy as np

In [2]:
# Carregando as bases

df_cadastros = pd.read_excel(r"cadastros.xlsx")
df_atendimentos = pd.read_excel(r"atendimentos_12_meses.xlsx")

#Visualizando dados de atendimentos

#df_atendimentos.head()
#df_cadastros.head()

Calculando o volume de atendimentos por paciente


In [3]:
# Agrupando os dados pelo ID

atendimentos_por_pacientes = df_atendimentos.groupby('ID_Vida').size().reset_index(name='Total_Atendimentos')

In [4]:
# Definindo o critério de 'High User'

media_atendimentos = atendimentos_por_pacientes['Total_Atendimentos'].mean()
desvio_padrao = atendimentos_por_pacientes['Total_Atendimentos'].std()

limite_high_user = np.ceil(media_atendimentos + (2 * desvio_padrao))


print(f"Média: {media_atendimentos:.2f} | Desvio: {desvio_padrao:.2f}")
print(f"Novo Critério: Pacientes com {limite_high_user} ou mais atendimentos são High Users.")

Média: 2.25 | Desvio: 3.02
Novo Critério: Pacientes com 9.0 ou mais atendimentos são High Users.


In [5]:
# Flagando os pacientes definidos como 'High Users'

atendimentos_por_pacientes['Flag_High_User'] = atendimentos_por_pacientes['Total_Atendimentos'].apply(
    lambda x: 'Sim' if x >= limite_high_user else 'Não')

In [6]:
# Cruzando as bases
# O'left' foi adotado para manter todos os beneficiários

df_final = pd.merge(df_cadastros, atendimentos_por_pacientes, on='ID_Vida', how='left')

In [7]:
# Tratamento de Nulos (Quem não teve atendimento fica com 0)

df_final['Total_Atendimentos'] = df_final['Total_Atendimentos'].fillna(0)
df_final['Flag_High_User'] = df_final['Flag_High_User'].fillna('Não')

In [8]:
# Exportar a base "mastigada" para o Power BI
df_final.to_excel('Base_Tratada.xlsx', index=False)